In [1]:
pip install noisereduce

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
# Path makes it easier to work with files and directories
# across different operating systems.


import librosa
# Librosa is used to load, resample, and process audio.


import soundfile as sf
# SoundFile is used to save the processed audio back to disk.


import numpy as np
# NumPy provides numerical operations used here for
# calculating the maximum audio amplitude.
import noisereduce as nr

# ============================================================
# AUDIO CONFIGURATION
# ============================================================

TARGET_SR = 16000
# Target sample rate = 16,000 Hz.
#
# This means every processed audio file will contain
# 16,000 audio samples per second.
#
# 16 kHz is commonly used for speech-processing models
# such as Whisper and WavLM.
NOISE_REDUCTION_STRENGTH = 0.7

MAX_DURATION = 120.0
# Maximum allowed audio duration = 10 seconds.
#
# Audio longer than 10 seconds will be skipped.

TARGET_SAMPLE_RATE = 16000
MIN_DURATION = 0.0
# Minimum allowed audio duration = 0.5 seconds.
#
# Very short audio clips are skipped because they may not
# contain enough useful speech information.


# ============================================================
# LOAD AUDIO
# ============================================================

def load_audio(audio_path, sr=TARGET_SR):
    """
    Load audio as mono and resample to the target sample rate.
    """
    # This function loads one audio file and prepares it
    # for the rest of the preprocessing pipeline.


    audio, _ = librosa.load(
        str(audio_path),
        sr=sr,
        mono=True
    )
    # librosa.load() reads the audio file.
    #
    # str(audio_path):
    # Converts a Path object into a string.
    #
    # sr=sr:
    # Resamples the audio to 16 kHz.
    #
    # mono=True:
    # Converts stereo/multi-channel audio into a single
    # mono channel.
    #
    # The "_" represents the original sample rate.
    # We don't need it because librosa has already resampled
    # the audio to our target sample rate.


    return audio, sr
    # Returns the processed audio waveform as a NumPy array.

# ============================================================
# REDUCE BACKGROUND NOISE
# ============================================================

def reduce_noise(audio, sr=TARGET_SAMPLE_RATE):

    # Avoid processing empty or extremely short audio.
    if len(audio) == 0:
        return audio

    # Apply spectral noise reduction.
    # stationary=False is suitable for noise that changes
    # over time. It does not require a separate noise recording.
    cleaned_audio = nr.reduce_noise(
        y=audio,
        sr=sr,
        stationary=False,
        prop_decrease=NOISE_REDUCTION_STRENGTH
    )

    return cleaned_audio
# ============================================================
# TRIM SILENCE
# ============================================================

def trim_silence(audio):
    """
    Remove leading and trailing silence.
    """
    # Removes unnecessary silence from the beginning
    # and end of the recording.


    trimmed, _ = librosa.effects.trim(
        audio,
        top_db=30
    )
    # librosa.effects.trim() detects regions containing
    # meaningful audio and removes surrounding silence.
    #
    # top_db=30 means audio below approximately 30 dB
    # relative to the peak is treated as silence.
    #
    # Example:
    #
    # silence + speech + silence
    #
    # becomes:
    #
    # speech


    return trimmed
    # Returns the trimmed waveform.


# ============================================================
# NORMALIZE AUDIO
# ============================================================


def normalize_audio(audio):

    if audio is None or len(audio) == 0:
        return audio

    max_value = np.max(np.abs(audio))

    if max_value > 0:
        audio = audio / max_value

    return audio



# ============================================================
# VALIDATE AUDIO
# ============================================================

def validate_audio(audio, sr=TARGET_SR):
    """
    Check whether audio duration is acceptable.
    """
    # Determines whether the processed audio is within
    # our minimum and maximum duration limits.


    duration = len(audio) / sr
    # Calculates duration in seconds.
    #
    # len(audio):
    # Number of audio samples.
    #
    # sr:
    # Samples per second.
    #
    # Example:
    #
    # 160,000 samples / 16,000 samples/sec
    # = 10 seconds


    return MIN_DURATION <= duration <= MAX_DURATION
    # Returns True if:
    #
    # 0.5 seconds <= duration <= 10 seconds
    #
    # Otherwise returns False.


# ============================================================
# PREPROCESS ONE AUDIO FILE
# ============================================================


def preprocess_audio(input_path, output_path):

    # Step 1: Load audio.
    audio, sr = load_audio(input_path)

    # Skip files that contain no samples.
    if audio is None or len(audio) == 0:
        print(f"Skipping empty audio: {input_path}")
        return False

    # Step 2: Reduce background noise.
    audio = reduce_noise(audio, sr=sr)

    # Check again after noise reduction.
    if audio is None or len(audio) == 0:
        print(f"Skipping empty audio after denoising: {input_path}")
        return False

    # Step 3: Trim leading and trailing silence.
    audio = trim_silence(audio)

    # Check again after trimming.
    if audio is None or len(audio) == 0:
        print(f"Skipping empty audio after trimming: {input_path}")
        return False

    # Step 4: Validate duration before normalization.
    if not validate_audio(audio, sr=sr):
        print(f"Skipping audio with invalid duration: {input_path}")
        return False

    # Step 5: Normalize audio.
    audio = normalize_audio(audio)

    # Step 6: Save processed audio as WAV.
    output_path = Path(output_path).with_suffix(".mp3")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    sf.write(
        str(output_path),
        audio,
        sr
    )

    return True



# ============================================================
# PROCESS DIRECTORY
# ============================================================

def process_directory(input_dir, output_dir):
    """
    Process all MP3 files in a directory recursively.
    """
    # Processes an entire directory rather than just
    # one audio file.


    input_dir = Path(input_dir)
    # Converts the input directory into a Path object.


    output_dir = Path(output_dir)
    # Converts the output directory into a Path object.


    wav_files = list(input_dir.rglob("*.mp3"))
    # Recursively searches the input directory for MP3 files.
    #
    # IMPORTANT:
    # The variable is named wav_files, but these are actually
    # MP3 files.
    #
    # A clearer name would be:
    #
    # audio_files = list(input_dir.rglob("*.mp3"))
    #
    # This does NOT affect functionality; it's just naming.


    processed = 0
    # Counts successfully processed audio files.


    skipped = 0
    # Counts audio files that were skipped or failed.


    for wav_file in wav_files:
        # Processes each MP3 file one at a time.


        relative_path = wav_file.relative_to(input_dir)
        # Calculates the file's location relative to the
        # input directory.
        #
        # Example:
        #
        # input:
        # commonvoice/train/audio/abc123.mp3
        #
        # relative path:
        # train/audio/abc123.mp3


        output_path = output_dir / relative_path
        # Recreates the same folder structure inside
        # the processed output directory.
        #
        # Example:
        #
        # ../data/processed/commonvoice/
        # train/audio/abc123.mp3


        try:
            # Use try/except so one bad audio file doesn't
            # stop the entire preprocessing job.


            success = preprocess_audio(
                wav_file,
                output_path
            )
            # Sends the current MP3 file through:
            #
            # 1. Load
            # 2. Convert to mono
            # 3. Resample to 16 kHz
            # 4. Trim silence
            # 5. Normalize
            # 6. Validate duration
            # 7. Save


            if success:
                # If preprocessing completed successfully...


                processed += 1
                # Increase the successful-processing count.


            else:
                # If validation failed...


                skipped += 1
                # Increase the skipped count.


        except Exception as e:
            # If an unexpected error occurs while processing
            # this particular audio file...


            print(
                f"Error processing {wav_file}: {e}"
            )
            # Print the filename and the error message.
            #
            # This makes it easier to troubleshoot corrupted
            # or unsupported audio files.


            skipped += 1
            # Count the failed file as skipped.


    print(f"Processed: {processed}")
    # Displays the number of successfully processed files.


    print(f"Skipped: {skipped}")
    # Displays the number of skipped/failed files.


    return processed, skipped
    # Returns both counts so another Python program could
    # use these values if needed.


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    # This block runs when the Python file is executed directly.
    #
    # It will not automatically run if this script is imported
    # into another Python file.


    REAL_INPUT = "../data/raw/commonvoice"
    # Location of the original Common Voice dataset.
    #
    # These files are your REAL speech samples.


    REAL_OUTPUT = "../data/processed/commonvoice"
    # Location where the cleaned Common Voice audio will
    # be saved.
    #
    # These processed files will become your REAL audio
    # samples for the voice-cloning detection dataset.


    process_directory(
        REAL_INPUT,
        REAL_OUTPUT
    )
    # Starts the preprocessing pipeline.
    #
    # RAW COMMON VOICE
    #       ↓
    # Load audio
    #       ↓
    # Convert to mono
    #       ↓
    # Resample to 16 kHz
    #       ↓
    # Trim silence
    #       ↓
    # Normalize
    #       ↓
    # Validate 0.5–10 sec
    #       ↓
    # PROCESSED COMMON VOICE



/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-45219.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-43251.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-45193.mp3


/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/noisereduce/spectralgate/nonstationary.py:70: RuntimeWarning: invalid value encountered in true_divide
  sig_mult_above_thresh = (abs_sig_stft - sig_stft_smooth) / sig_stft_smooth


Skipping empty audio after trimming: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-20309.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-43241.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-43269.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-43233.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-43230.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-meh/audios/spontaneous-speech-meh-43225.mp3
Skipping empty audio after trimming: ../data/raw/commonvoic

/var/folders/0g/jtvhdh2x0xl58z5r29424k1m0000gn/T/ipykernel_93222/3591889073.py:58: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(
/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error processing ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-kcn/audios/spontaneous-speech-kcn-23643.mp3: 
Error processing ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-kcn/audios/spontaneous-speech-kcn-22681.mp3: 
Error processing ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-kcn/audios/spontaneous-speech-kcn-22647.mp3: 
Error processing ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-kcn/audios/spontaneous-speech-kcn-23464.mp3: 
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-sco/audios/spontaneous-speech-sco-45812.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-sco/audios/spontaneous-speech-sco-45755.mp3
Skipping audio with invalid duration: ../data/raw/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-sco/audios/spontaneou

KeyboardInterrupt: 